In [1]:
import fastf1
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

DATA_DIR   = Path("~/Downloads/F1 race intelligence/data").expanduser()
MODELS_DIR = Path("~/Downloads/F1 race intelligence/models").expanduser()
CACHE_DIR  = Path.home() / "Library/Caches/fastf1"

fastf1.Cache.enable_cache(str(CACHE_DIR))
print("✅ Ready")

✅ Ready


In [2]:
# ── Load all race sessions with telemetry ─────────────────
SEASONS = range(2018, 2027)

STYLE_FEATURES = [
    'brake_aggression',
    'throttle_smoothness', 
    'throttle_attack',
    'top_speed',
    'coasting_ratio',
    'gear_aggression',
    'consistency',
    'tyre_management'
]

def extract_driver_features(session, driver):
    """Extract style features for one driver from one session."""
    feats = {}
    laps  = session.laps.copy()
    laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
    clean = laps[
        (laps['LapTimeSeconds'] > 60) &
        (laps['LapTimeSeconds'] < 200) &
        (laps['IsAccurate'] == True) &
        (laps['Driver'] == driver)
    ].copy()

    if len(clean) < 5:
        return None

    # ── Telemetry features ────────────────────────────────
    try:
        fastest_lap = session.laps.pick_drivers(driver).pick_fastest()
        tel = fastest_lap.get_car_data().add_distance()

        if len(tel) > 10:
            braking = tel[tel['Brake'] == True]
            feats['brake_aggression'] = float(abs(braking['Speed'].diff().dropna().mean())) if len(braking) > 5 else 0.0
            feats['throttle_smoothness'] = float(tel['Throttle'].std())
            accel = tel[tel['Speed'].diff() > 0]
            feats['throttle_attack'] = float(accel['Throttle'].mean()) if len(accel) > 5 else 50.0
            feats['top_speed'] = float(tel['Speed'].quantile(0.95))
            coasting = tel[(tel['Throttle'] < 5) & (tel['Brake'] == False)]
            feats['coasting_ratio'] = float(len(coasting) / len(tel) * 100)
            mid_speed = tel[(tel['Speed'] > 100) & (tel['Speed'] < 200)]
            feats['gear_aggression'] = float(mid_speed['nGear'].mean()) if len(mid_speed) > 5 else 5.0
        else:
            for f in ['brake_aggression','throttle_smoothness','throttle_attack',
                      'top_speed','coasting_ratio','gear_aggression']:
                feats[f] = np.nan
    except:
        for f in ['brake_aggression','throttle_smoothness','throttle_attack',
                  'top_speed','coasting_ratio','gear_aggression']:
            feats[f] = np.nan

    # ── Lap-level features ────────────────────────────────
    feats['consistency'] = float(clean['LapTimeSeconds'].std())
    early = clean[clean['TyreLife'] <= 5]['LapTimeSeconds'].mean()
    late  = clean[clean['TyreLife'] >= 15]['LapTimeSeconds'].mean()
    feats['tyre_management'] = float(late - early) if not np.isnan(early) and not np.isnan(late) else np.nan

    return feats

In [3]:
# ── Main collection loop ──────────────────────────────────
import time
from fastf1.exceptions import RateLimitExceededError

driver_race_features = {}  # {driver: [list of feature dicts per race]}
skipped = []

for year in SEASONS:
    try:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        completed = schedule[pd.to_datetime(schedule['EventDate']) < pd.Timestamp.now()]
        print(f"\n{'='*50}")
        print(f"📅 {year} — {len(completed)} races")
    except Exception as e:
        print(f"⚠️ {year}: {e}")
        continue

    for _, event in completed.iterrows():
        round_num  = int(event['RoundNumber'])
        event_name = event['EventName']
        print(f"  🏎️ {event_name}...", end='', flush=True)

        try:
            session = fastf1.get_session(year, round_num, 'R')
            session.load(laps=True, telemetry=True, weather=False, messages=False)

            for drv in session.laps['Driver'].unique():
                feats = extract_driver_features(session, drv)
                if feats:
                    feats['year']  = year
                    feats['round'] = round_num
                    feats['event'] = event_name
                    if drv not in driver_race_features:
                        driver_race_features[drv] = []
                    driver_race_features[drv].append(feats)

            print(f" ✅ {len(session.laps['Driver'].unique())} drivers")
            time.sleep(1)

        except RateLimitExceededError:
            print(f" ⏳ Rate limited — waiting 60s...")
            time.sleep(60)
            skipped.append((year, round_num))
        except Exception as e:
            print(f" ❌ {e}")
            skipped.append((year, round_num))

print(f"\n✅ Done — {len(driver_race_features)} drivers collected")
print(f"Skipped: {len(skipped)}")


📅 2018 — 21 races
  🏎️ Australian Grand Prix...

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 5 completed the race distance 00:00.123000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
core        WARNING 	Car position data is unavailable!
core           INFO 	Finished loading data for 20 drivers: ['5', '44', '7', '3', '14', '33', '27', '77', '2', '55', '11', '

 ✅ 20 drivers
  🏎️ Bahrain Grand Prix...

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
core        WARNING 	Car telemetry data is unavailable!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
core        WARNING 	Car position data is unavailable!
core        WARNING 	Failed to determine `Session.t0_date`!
logger      WARNING 	Failed to load telemetry data!
core   

 ✅ 20 drivers
  🏎️ Chinese Grand Prix...

...

 ✅ 22 drivers
  🏎️ Barcelona Grand Prix...

core           INFO 	Loading data for Barcelona Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
logger      WARNING 	Failed to load result data from Ergast!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data

 ✅ 22 drivers
  🏎️ Austrian Grand Prix...

core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No c

 ✅ 22 drivers

✅ Done — 44 drivers collected
Skipped: 1


In [4]:
# ── Aggregate per driver ──────────────────────────────────
driver_profiles = []

for drv, races in driver_race_features.items():
    profile = {
        'driver': drv,
        'races':  len(races),
        'seasons': sorted(list(set(r['year'] for r in races)))
    }
    for feat in STYLE_FEATURES:
        vals = [r[feat] for r in races if not np.isnan(r.get(feat, np.nan))]
        profile[feat] = float(np.mean(vals)) if vals else 0.0
    driver_profiles.append(profile)

print(f"✅ {len(driver_profiles)} driver profiles built")
for p in sorted(driver_profiles, key=lambda x: x['driver']):
    print(f"  {p['driver']}: {p['races']} races, {p['seasons']}")

✅ 44 driver profiles built
  AIT: 1 races, [2020]
  ALB: 128 races, [2019, 2020, 2022, 2023, 2024, 2025, 2026]
  ALO: 136 races, [2018, 2021, 2022, 2023, 2024, 2025, 2026]
  ANT: 30 races, [2025, 2026]
  BEA: 35 races, [2024, 2025, 2026]
  BOR: 28 races, [2025, 2026]
  BOT: 150 races, [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
  COL: 33 races, [2024, 2025, 2026]
  DEV: 11 races, [2022, 2023]
  DOO: 5 races, [2024, 2025]
  ERI: 19 races, [2018]
  FIT: 2 races, [2020]
  GAS: 166 races, [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
  GIO: 58 races, [2019, 2020, 2021]
  GRO: 52 races, [2018, 2019, 2020]
  HAD: 29 races, [2025, 2026]
  HAM: 175 races, [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
  HAR: 17 races, [2018]
  HUL: 116 races, [2018, 2019, 2020, 2022, 2023, 2024, 2025, 2026]
  KUB: 23 races, [2019, 2021]
  KVY: 37 races, [2019, 2020]
  LAT: 58 races, [2020, 2021, 2022]
  LAW: 40 races, [2023, 2024, 2025, 2026]
  LEC: 170 races, [2018, 2019, 2020, 2021, 202

In [5]:
# ── Fit PCA + Scaler on ALL historical drivers ────────────
feat_matrix = np.array([[p[f] for f in STYLE_FEATURES] for p in driver_profiles])

# Impute NaN with column means
col_means = np.nanmean(feat_matrix, axis=0)
for i in range(feat_matrix.shape[1]):
    feat_matrix[np.isnan(feat_matrix[:, i]), i] = col_means[i]

# Fit scaler + PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feat_matrix)

pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

print(f"✅ PCA variance explained: {pca.explained_variance_ratio_}")

# Save PCA + Scaler — used to PROJECT new drivers in
joblib.dump(scaler, MODELS_DIR / "driver_embedding_scaler.pkl")
joblib.dump(pca,    MODELS_DIR / "driver_embedding_pca.pkl")
print("✅ PCA + Scaler saved")

✅ PCA variance explained: [0.26956162 0.2433573 ]
✅ PCA + Scaler saved


In [6]:
# ── Compute similarity matrix ─────────────────────────────
sim_matrix = cosine_similarity(X_scaled)
driver_names = [p['driver'] for p in driver_profiles]

# Team colors
TEAM_COLORS = {
    'Red Bull Racing': '#3671C6', 'Ferrari': '#E8002D',
    'Mercedes': '#27F4D2', 'McLaren': '#FF8000',
    'Aston Martin': '#229971', 'Alpine': '#FF87BC',
    'Williams': '#64C4FF', 'RB': '#6692FF',
    'Racing Bulls': '#6692FF', 'Kick Sauber': '#52E252',
    'Haas F1 Team': '#B6BABD', 'AlphaTauri': '#5E8FAA',
    'Alfa Romeo': '#C92D4B', 'Renault': '#FFF500',
    'Force India': '#F596C8', 'Racing Point': '#F596C8',
    'Toro Rosso': '#469BFF', 'Sauber': '#C92D4B',
}

# Build final embedding database
embedding_db = []
for i, p in enumerate(driver_profiles):
    # Top 10 most similar
    sims = [(driver_names[j], round(float(sim_matrix[i][j]), 4))
            for j in range(len(driver_names)) if j != i]
    sims.sort(key=lambda x: x[1], reverse=True)

    embedding_db.append({
        'driver':   p['driver'],
        'races':    p['races'],
        'seasons':  p['seasons'],
        'x':        round(float(coords[i][0]), 4),
        'y':        round(float(coords[i][1]), 4),
        'features': {f: round(p[f], 4) for f in STYLE_FEATURES},
        'similar':  sims[:10],
        'color':    '#888888',  # updated at runtime based on current team
        'col_means': col_means.tolist(),  # needed for projection
    })

# Save embedding database
out = DATA_DIR / "driver_embeddings_alltime.json"
with open(out, 'w') as f:
    json.dump({
        'drivers':       embedding_db,
        'features':      STYLE_FEATURES,
        'pca_variance':  pca.explained_variance_ratio_.tolist(),
        'col_means':     col_means.tolist(),
        'total_drivers': len(embedding_db),
        'seasons':       list(range(2018, 2027)),
    }, f, indent=2)

print(f"✅ Saved {len(embedding_db)} driver embeddings → {out}")
print(f"File size: {out.stat().st_size / 1024:.1f} KB")

✅ Saved 44 driver embeddings → /Users/sanjaychowdary/Downloads/F1 race intelligence/data/driver_embeddings_alltime.json
File size: 58.1 KB
